# Herramienta 02 — Clasificación de Conducción Distractiva

Clasifica imágenes de conductores en 5 categorías usando **Transfer Learning con MobileNetV2** (aprendizaje profundo).

**Dataset:** [Multi-Class Driver Behavior Image Dataset](https://www.kaggle.com/datasets/arafatsahinafridi/multi-class-driver-behavior-image-dataset) — 7 276 imágenes reales, Bangladesh 2024.  
**Clases:** `safe_driving`, `texting_phone`, `talking_phone`, `turning`, `other_activities`  
**Modelo:** MobileNetV2 preentrenado en ImageNet + fine-tuning en Colab T4 GPU (~15 min).

---
> **Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU (T4)**

## 1. Verificar GPU y versiones

In [ ]:
import tensorflow as tf
import numpy as np

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('SIN GPU — activa T4 en Entorno de ejecución -> Cambiar tipo.')
else:
    print('GPU OK:', gpus[0].name)

## 2. Dataset

**Fuente:** [Multi-Class Driver Behavior Image Dataset](https://www.kaggle.com/datasets/arafatsahinafridi/multi-class-driver-behavior-image-dataset) — Zenodo/Mendeley Data ([DOI: 10.17632/mzb4b6dff3.1](https://data.mendeley.com/datasets/mzb4b6dff3/1)). Recolectado en Ashulia, Dhaka (Bangladesh), octubre 2024, con cámaras de teléfono en vehículos privados y buses públicos.

Las imágenes se descargan desde el repositorio GitHub usando `sparse-checkout` (solo la carpeta de imágenes, ~33 MB). No se requiere cuenta de Kaggle.

**Limitación conocida:** el dataset no incluye clase de somnolencia. Conductas de sueño al volante quedan dentro de `other_activities`.

In [ ]:
import subprocess, os
from pathlib import Path
import pandas as pd

REPO    = 'https://github.com/AndresGuido9820/sistema-transporte-inteligente'
REPO_DIR = 'sistema-transporte-inteligente'

if not Path(REPO_DIR).exists():
    print('Descargando imágenes...')
    subprocess.run(['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse', REPO], check=True)
    subprocess.run(['git', 'sparse-checkout', 'set',
                    'data/processed/driver_images_real',
                    'data/processed/driver_images.csv'],
                   cwd=REPO_DIR, check=True)
else:
    print('Ya clonado.')

df = pd.read_csv(f'{REPO_DIR}/data/processed/driver_images.csv')
df['full_path'] = REPO_DIR + '/' + df['image_path']

CLASSES  = ['safe_driving', 'talking_phone', 'texting_phone', 'turning', 'other_activities']
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
df['label_idx'] = df['label'].map(CLASS_IDX)
df = df.dropna(subset=['label_idx'])
df['label_idx'] = df['label_idx'].astype('int32')

print(f'Total imágenes: {len(df)}')
display(df['label'].value_counts())

## 3. Exploración visual — ejemplos por clase

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 14))
for row, cls in enumerate(CLASSES):
    samples = df[df['label'] == cls].sample(4, random_state=42)
    for col, (_, s) in enumerate(samples.iterrows()):
        axes[row, col].imshow(Image.open(s['full_path']).convert('RGB'))
        axes[row, col].set_title(cls if col == 0 else '', fontsize=9, fontweight='bold')
        axes[row, col].axis('off')
plt.suptitle('4 ejemplos por clase del dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Partición del dataset

Partición estratificada por clase: **70% train / 15% validación / 15% test**.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
IMAGE_SIZE = (96, 96)
BATCH = 32

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df,   test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nDistribución train:')
print(train_df['label'].value_counts().to_string())

## 5. Pipeline de datos con tf.data

Se aplica **data augmentation** al conjunto de entrenamiento (flip horizontal, brillo, contraste, saturación) para mejorar la generalización. MobileNetV2 espera entradas en rango `[-1, 1]`.

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)  # [-1, 1]
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img, label

def make_ds(dataframe, shuffle=False, do_augment=False):
    paths  = dataframe['full_path'].values
    labels = dataframe['label_idx'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if do_augment:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, shuffle=True, do_augment=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

print('Pipelines listos.')
print(f'Batches — train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}')

## 6. Modelo — MobileNetV2 con transfer learning

**Arquitectura:**
```
MobileNetV2 (ImageNet, base congelada)
  └── GlobalAveragePooling2D
  └── Dense(256, relu)
  └── Dropout(0.4)
  └── Dense(128, relu)
  └── Dropout(0.3)
  └── Dense(5, softmax)
```

**Por qué MobileNetV2:** diseñada para imágenes pequeñas (funciona bien a 96×96), pesa ~14 MB, se descarga automáticamente de imagenet, y el fine-tuning en Colab T4 tarda ~15 min.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False  # congelado en fase 1

inputs  = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x       = base(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(256, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.4)(x)
x       = tf.keras.layers.Dense(128, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

total     = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Parámetros totales   : {total:,}')
print(f'Entrenables (fase 1) : {trainable:,}')

## 7. Fase 1 — Entrenamiento de la cabeza (base congelada)

Solo se actualizan las capas Dense añadidas. ~5 epochs, ~2 min en T4.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print('=== FASE 1: cabeza ===')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=cb1)
print(f'Val accuracy fase 1: {max(h1.history["val_accuracy"]):.4f}')

## 8. Fase 2 — Fine-tuning (últimas 30 capas de MobileNetV2)

Se descongela el 20% final de la base con lr 10× menor para adaptar los filtros al dominio de conducción sin destruir los pesos preentrenados. ~10–15 epochs, ~10 min en T4.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

trainable2 = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Parámetros entrenables (fase 2): {trainable2:,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.ModelCheckpoint('mejor_modelo.keras', monitor='val_accuracy',
                                        save_best_only=True, verbose=1)
]

print('=== FASE 2: fine-tuning ===')
h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cb2)

## 9. Curvas de entrenamiento

In [ ]:
acc     = h1.history['accuracy']     + h2.history['accuracy']
val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss    = h1.history['loss']         + h2.history['loss']
val_los = h1.history['val_loss']     + h2.history['val_loss']
sep     = len(h1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, train_v, val_v, title in [
    (axes[0], acc, val_acc, 'Accuracy'),
    (axes[1], loss, val_los, 'Loss')
]:
    ax.plot(train_v, label='Train', color='#2f6fff')
    ax.plot(val_v,   label='Val',   color='#11a87d')
    ax.axvline(sep, color='#f6a531', linestyle='--', label='Fine-tune')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Época')
    ax.legend(); ax.grid(alpha=0.25)

plt.suptitle('MobileNetV2 — Fase 1 (cabeza) + Fase 2 (fine-tuning)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Evaluación en test — Métricas y matriz de confusión

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true, y_pred_list = [], []
for imgs, labels in test_ds:
    probs = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_list.extend(np.argmax(probs, axis=1))

y_true      = np.array(y_true)
y_pred_arr  = np.array(y_pred_list)

_, acc_test = model.evaluate(test_ds, verbose=0)
print(f'Accuracy en test: {acc_test:.4f}\n')
print(classification_report(y_true, y_pred_arr, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred_arr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Matriz de Confusión (test accuracy {acc_test:.2%})', fontweight='bold')
plt.ylabel('Real'); plt.xlabel('Predicho')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 11. Ejemplos: aciertos y errores del modelo

In [ ]:
# Predicciones ordenadas (sin shuffle)
test_ds_ord = make_ds(test_df, shuffle=False)
all_probs, all_lbls = [], []
for imgs, lbls in test_ds_ord:
    all_probs.append(model.predict(imgs, verbose=0))
    all_lbls.extend(lbls.numpy())
all_probs = np.concatenate(all_probs, axis=0)
all_lbls  = np.array(all_lbls)
pred_ord  = np.argmax(all_probs, axis=1)
paths_ord = test_df['full_path'].values

hits   = np.where(pred_ord == all_lbls)[0]
misses = np.where(pred_ord != all_lbls)[0]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Aciertos (verde) y Errores (rojo) del modelo', fontsize=13, fontweight='bold')

for col in range(5):
    for row, indices, color in [(0, hits, 'green'), (1, misses, 'red')]:
        if col < len(indices):
            i = indices[col]
            axes[row, col].imshow(Image.open(paths_ord[i]).convert('RGB'))
            axes[row, col].set_title(
                f'Real: {CLASSES[all_lbls[i]]}\nPred: {CLASSES[pred_ord[i]]}\n({all_probs[i, pred_ord[i]]:.0%})',
                fontsize=8, color=color)
        axes[row, col].axis('off')

plt.tight_layout(); plt.show()

## 12. Análisis de distracciones — Informe preventivo

In [ ]:
from sklearn.metrics import f1_score

class_counts = df['label'].value_counts().reindex(CLASSES)
class_pct    = class_counts / class_counts.sum() * 100
f1s          = f1_score(y_true, y_pred_arr, average=None)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['#11a87d','#2f6fff','#f6a531','#e06d2f','#8a5cf5']

b1 = axes[0].bar(CLASSES, class_pct, color=colors)
for b, v in zip(b1, class_pct):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f'{v:.1f}%', ha='center', fontsize=9)
axes[0].set_title('Distribución de clases en el dataset', fontweight='bold')
axes[0].set_ylabel('% del total'); axes[0].grid(axis='y', alpha=0.25)
axes[0].tick_params(axis='x', rotation=25)

b2 = axes[1].bar(CLASSES, f1s, color=colors)
for b, v in zip(b2, f1s):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.2f}', ha='center', fontsize=9)
axes[1].axhline(np.mean(f1s), color='black', linestyle='--', linewidth=1,
                label=f'Macro avg {np.mean(f1s):.2f}')
axes[1].set_title('F1-score por clase (test)', fontweight='bold')
axes[1].set_ylabel('F1'); axes[1].set_ylim(0, 1.15)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.25)
axes[1].tick_params(axis='x', rotation=25)

plt.suptitle('Análisis de tipos de distracción', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n=== Medidas preventivas por tipo de distracción ===')
print('  texting_phone    -> Bloqueador de pantalla automático a >20 km/h.')
print('  talking_phone    -> Manos libres obligatorio; alerta al supervisor >3 s.')
print('  turning          -> Formación en señalización y uso de espejos.')
print('  other_activities -> Pausas activas cada 2 h; revisión de ergonomía.')
print('  (somnolencia)    -> No cubierta — requiere dataset adicional (MRL/DDD).')

## 13. Exportar modelo para la herramienta web

Se guardan dos archivos:
- **`modelo_conduccion_cnn.keras`** — formato Keras nativo (para reutilizar en Python)
- **`modelo_conduccion.tflite`** — cuantizado, archivo único, para la página web con TF.js

Pasa el `.tflite` después de descargarlo para actualizar la página web.

In [ ]:
from google.colab import files

# Keras nativo
model.save('modelo_conduccion_cnn.keras')
print('Guardado: modelo_conduccion_cnn.keras')

# TFLite cuantizado (reduce tamaño ~4×)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

with open('modelo_conduccion.tflite', 'wb') as f:
    f.write(tflite_bytes)

import os
mb = os.path.getsize('modelo_conduccion.tflite') / 1e6
print(f'Guardado: modelo_conduccion.tflite ({mb:.1f} MB)')

files.download('modelo_conduccion_cnn.keras')
files.download('modelo_conduccion.tflite')
print('Descarga iniciada.')

## 14. Herramienta — clasificar imagen nueva

In [ ]:
import io
uploaded = files.upload()

for fname, data in uploaded.items():
    img_pil = Image.open(io.BytesIO(data)).convert('RGB').resize(IMAGE_SIZE)
    arr = np.array(img_pil, dtype=np.float32)
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    arr = np.expand_dims(arr, 0)

    probs = model.predict(arr, verbose=0)[0]
    idx   = int(np.argmax(probs))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(Image.open(io.BytesIO(data)))
    axes[0].set_title(f'Predicción: {CLASSES[idx]}\nConfianza: {probs[idx]:.1%}',
                      fontweight='bold', fontsize=12); axes[0].axis('off')

    bar_colors = ['#11a87d' if i == idx else '#c9d8ff' for i in range(len(CLASSES))]
    axes[1].barh(CLASSES, probs, color=bar_colors)
    axes[1].set_xlim(0, 1)
    axes[1].set_title('Probabilidad por clase', fontweight='bold')
    axes[1].set_xlabel('Probabilidad'); axes[1].grid(axis='x', alpha=0.25)
    for i, p in enumerate(probs):
        axes[1].text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=9)

    plt.tight_layout(); plt.show()

## 15. Conclusiones

MobileNetV2 con fine-tuning aplica **aprendizaje profundo** real al problema de clasificación visual de conducción distractiva. Las capas convolucionales preentrenadas en ImageNet detectan bordes, texturas y formas de alto nivel (postura de manos, posición del teléfono, ángulo de la cabeza) que los descriptores manuales HOG no capturan.

**Limitaciones:**
- Sin clase de somnolencia (la más crítica en transporte de pasajeros)
- Dataset de Bangladesh — generalización a condiciones latinoamericanas no validada
- Clasificación por fotograma: en producción se requieren secuencias de ≥3 s para evitar falsas alertas

**Trabajo futuro:** recolección local con consentimiento, adición de somnolencia (dataset MRL), despliegue en borde con TFLite Runtime en Raspberry Pi 4 para monitoreo a bordo en tiempo real.